# Лекция и методическое руководство: Подзапросы и операции над множествами в SQL
## Конспект преподавателя (для объяснения у доски и живой демонстрации)

> **Как пользоваться этим блокнотом на паре:**
> * Блоки **«📌 На доску»** содержат емкие синтаксические схемы и правила для записи на доске мелом/маркером.
> * Блоки **«💡 Зачем это нужно»** дают понятные жизненные аналогии для объяснения логического смысла студентам.
> * Блоки **«⚙️ Как это работает»** раскрывают физику выполнения запроса СУБД (трехзначная логика, short-circuit, корреляция).
> * Блоки **«💻 Демонстрация»** содержат живой рабочий код для запуска на проекторе в интерактивной базе данных.
> * Блоки **«⚠️ Грабли новичков»** разбирают коварные ошибки (ловушка `NOT IN + NULL`, отсутствие алиаса в `FROM`, подзапрос в `SELECT` с несколькими строками).

---

### План занятия (тайминг на 90 минут):
1. **Анатомия подзапросов в SQL и места их применения** (10 мин)
2. **Скалярные подзапросы в `WHERE` и операторы сравнения (`=`, `>`, `<`, `<>`)** (10 мин)
3. **Множественные подзапросы (`IN`)** (10 мин)
4. **Смертельная ловушка трехзначной логики: `NOT IN` и `NULL`** (15 мин)
5. **Коррелированные подзапросы (сравнение с контекстом группы)** (15 мин)
6. **Предикаты существования `EXISTS` и `NOT EXISTS` (быстро и безопасно)** (10 мин)
7. **Подзапросы в `SELECT` (витрина «Клиент 360») и производные таблицы в `FROM`** (10 мин)
8. **Операции над множествами (`UNION`, `UNION ALL`, `INTERSECT`, `EXCEPT`)** (10 мин)


In [1]:
import sqlite3
import pandas as pd

# Создаем базу данных в оперативной памяти для живых демонстраций
conn = sqlite3.connect(":memory:")
conn.execute("PRAGMA foreign_keys = ON;")
cur = conn.cursor()

# Создаем взаимосвязанную схему киберспортивного клуба
cur.executescript('''
CREATE TABLE demo_members (
    member_id INTEGER PRIMARY KEY,
    nickname TEXT NOT NULL,
    city TEXT NOT NULL,
    loyalty_tier TEXT NOT NULL
);

CREATE TABLE demo_pcs (
    pc_id INTEGER PRIMARY KEY,
    zone_name TEXT NOT NULL,
    hourly_rate INTEGER NOT NULL
);

CREATE TABLE demo_sessions (
    session_id INTEGER PRIMARY KEY,
    member_id INTEGER REFERENCES demo_members(member_id),
    pc_id INTEGER REFERENCES demo_pcs(pc_id),
    game_title TEXT NOT NULL,
    hours_played REAL NOT NULL,
    total_paid INTEGER NOT NULL
);

CREATE TABLE demo_bar_orders (
    order_id INTEGER PRIMARY KEY,
    member_id INTEGER REFERENCES demo_members(member_id),
    item_name TEXT NOT NULL,
    category TEXT NOT NULL,
    total_amount INTEGER NOT NULL
);

-- Наполняем тестовыми данными со специальными краевыми случаями (NULL, неактивные гости)
INSERT INTO demo_members VALUES
(1, 'Shadow', 'Москва', 'Gold'),
(2, 'Viper', 'Санкт-Петербург', 'Silver'),
(3, 'Neo', 'Москва', 'Platinum'),
(4, 'Ghost', 'Казань', 'Bronze'),
(5, 'Frost', 'Москва', 'Bronze'),      -- Зарегистрирован, но ни разу не играл и не был в баре
(6, 'Phoenix', 'Самара', 'Silver');    -- Только играл, но в бар ни ногой!

INSERT INTO demo_pcs VALUES
(101, 'Standard', 150),
(102, 'Standard', 150),
(103, 'VIP', 400),
(104, 'Bootcamp', 250); -- Простаивающий компьютер без единой сессии

INSERT INTO demo_sessions VALUES
(1, 1, 101, 'Dota 2', 2.0, 300),
(2, 1, 103, 'Cyberpunk 2077', 3.0, 1200),
(3, 2, 101, 'Dota 2', 1.0, 150),
(4, 2, 102, 'Counter-Strike 2', 4.0, 600),
(5, 3, 103, 'Counter-Strike 2', 2.5, 1000),
(6, 6, 101, 'Counter-Strike 2', 5.0, 750),
(7, NULL, 102, 'Dota 2', 1.5, 225); -- Анонимный гость без клубной карты!

INSERT INTO demo_bar_orders VALUES
(1, 1, 'Энергетик Red Bull', 'Напитки', 250),
(2, 1, 'Сэндвич с ветчиной', 'Горячее', 350),
(3, 2, 'Чипсы Lays', 'Снеки', 180),
(4, 3, 'Энергетик Flash', 'Напитки', 150),
(5, 3, 'Пицца пепперони', 'Горячее', 650),
(6, NULL, 'Кофе Американо', 'Напитки', 120); -- Анонимный заказ в баре (NULL member_id)!
''')
conn.commit()

print("✅ Демонстрационная база данных с краевыми случаями развернута в памяти.")


✅ Демонстрационная база данных с краевыми случаями развернута в памяти.


---
## Тема 1. Анатомия подзапросов в SQL

### 📌 На доску
```
                       ┌─────────────────────────────────────────┐
                       │          ПОДЗАПРОСЫ В SQL (SUBQUERIES)  │
                       └─────────────────────────────────────────┘
                                            │
        ┌───────────────────────────────────┼──────────────────────────────────┐
        ▼                                   ▼                                  ▼
 ┌───────────────┐                   ┌───────────────┐                  ┌───────────────┐
 │   В SELECT    │                   │    В WHERE    │                  │    В FROM     │
 │ (Вычисленный  │                   │ (Фильтрация   │                  │ (Производная  │
 │    столбец)   │                   │  по условию)  │                  │   таблица)    │
 └───────────────┘                   └───────────────┘                  └───────────────┘
        │                                   │                                  │
 Требует СТРОГО                      ┌──────┴──────┐                     ОБЯЗАТЕЛЕН
 1 строку и 1 колонку                ▼             ▼                     псевдоним AS!
                             Скалярный       Множественный
                            (=, >, <, <>)    (IN, EXISTS)
```

### 💡 Зачем это нужно
* В реальной аналитике мы часто не знаем точного числа заранее. Мы не можем написать:  
  `WHERE total_paid > 500`, потому что средний чек меняется каждую минуту.
* Подзапрос позволяет СУБД **самостоятельно вычислить критерий** и сразу же применить его для фильтрации данных.


---
## Тема 2. Скалярные подзапросы в `WHERE` и сравнения (`=`, `>`, `<`, `<>`)

### 📌 На доску
```sql
-- Найти сессии, которые дороже среднего чека по клубу
SELECT session_id, game_title, total_paid
FROM demo_sessions
WHERE total_paid > (SELECT AVG(total_paid) FROM demo_sessions);
```

### ⚙️ Как это работает
1. СУБД сначала выполняет внутренний подзапрос: `SELECT AVG(total_paid) FROM demo_sessions` $	o$ получается скалярное число (например, `603.57`).
2. Внутренний запрос выполняется **один раз** (некоррелированный).
3. Затем СУБД фильтрует внешнюю таблицу: `WHERE total_paid > 603.57`.

### ⚠️ Грабли новичков
* Если подзапрос вернет **больше одного значения** (например, забыли `AVG()` или написали `SELECT total_paid FROM demo_sessions`), СУБД SQLite выдаст ошибку:  
  `OperationalError: subquery returns more than 1 row`.


In [2]:
# Демонстрация 2: Скалярный подзапрос в WHERE
query_scalar = '''
SELECT 
    session_id,
    game_title,
    hours_played,
    total_paid,
    (SELECT ROUND(AVG(total_paid), 2) FROM demo_sessions) AS club_avg_paid
FROM demo_sessions
WHERE total_paid > (SELECT AVG(total_paid) FROM demo_sessions)
ORDER BY total_paid DESC;
'''
pd.read_sql_query(query_scalar, conn)


session_id,game_title,hours_played,total_paid,club_avg_paid
2,Cyberpunk 2077,3.0,1200,603.57
5,Counter-Strike 2,2.5,1000,603.57
6,Counter-Strike 2,5.0,750,603.57


---
## Тема 3. Множественные подзапросы: оператор `IN`

### 📌 На доску
```sql
-- Найти клиентов, игравших в Counter-Strike 2
SELECT member_id, nickname, city
FROM demo_members
WHERE member_id IN (
    SELECT member_id 
    FROM demo_sessions 
    WHERE game_title = 'Counter-Strike 2'
);
```

### 💡 Зачем это нужно
* Когда критерий отбора — это не одно число, а **список идентификаторов** из связанной таблицы.
* Внутренний запрос возвращает один столбец с набором значений `(2, 3, 6)`.
* Внешний запрос оставляет только тех клиентов, чей `member_id` входит в этот список.


In [3]:
# Демонстрация 3: Подзапрос с оператором IN
query_in = '''
SELECT member_id, nickname, city, loyalty_tier
FROM demo_members
WHERE member_id IN (
    SELECT member_id 
    FROM demo_sessions 
    WHERE game_title = 'Counter-Strike 2'
);
'''
pd.read_sql_query(query_in, conn)


member_id,nickname,city,loyalty_tier
2,Viper,Санкт-Петербург,Silver
3,Neo,Москва,Platinum
6,Phoenix,Самара,Silver


---
## Тема 4. Смертельная ловушка трехзначной логики: `NOT IN` и `NULL`

### 📌 На доску
```sql
-- ОПАСНО! Если во вложенном запросе есть хоть один NULL — вернется 0 строк!
SELECT nickname FROM demo_members
WHERE member_id NOT IN (SELECT member_id FROM demo_bar_orders);

-- БЕЗОПАСНЫЙ СПОСОБ 1 (Явная фильтрация NULL):
SELECT nickname FROM demo_members
WHERE member_id NOT IN (SELECT member_id FROM demo_bar_orders WHERE member_id IS NOT NULL);

-- БЕЗОПАСНЫЙ СПОСОБ 2 (Стандарт индустрии: NOT EXISTS):
SELECT nickname FROM demo_members m
WHERE NOT EXISTS (SELECT 1 FROM demo_bar_orders b WHERE b.member_id = m.member_id);
```

### ⚙️ Как это работает (Физика трехзначной логики)
В реляционной алгебре SQL есть три логических состояния: `TRUE`, `FALSE`, `UNKNOWN`.
Выражение `x NOT IN (1, 2, NULL)` математически раскрывается через логическое `AND`:
$$x \neq 1 \quad \text{AND} \quad x \neq 2 \quad \text{AND} \quad x \neq \text{NULL}$$

* Сравнение любого значения с `NULL` (даже `5 <> NULL`) возвращает не `TRUE`, а **`UNKNOWN`**!
* В булевой алгебре: `TRUE AND TRUE AND UNKNOWN` $\equiv$ **`UNKNOWN`**.
* Условие `WHERE` пропускает строку **только если результат строго `TRUE`**. Значение `UNKNOWN` отбрасывается как `FALSE`.
* **Результат катастрофы:** если в таблице из 1 000 000 строк есть хотя бы одна строка с `NULL`, запрос `NOT IN` возвращает **пустое множество**!


In [4]:
# Демонстрация 4: Наглядный крах NOT IN из-за наличия анонимного заказа (NULL) в баре!

print("1. Опасный запрос (без фильтрации NULL) возвращает:")
query_broken = '''
SELECT member_id, nickname 
FROM demo_members
WHERE member_id NOT IN (SELECT member_id FROM demo_bar_orders);
'''
res_broken = pd.read_sql_query(query_broken, conn)
display(res_broken)
print(f"Строк возвращено: {len(res_broken)} (Хотя клиенты Ghost, Frost и Phoenix в баре ничего не брали!)")

print("\n2. Исправленный запрос с фильтром WHERE member_id IS NOT NULL:")
query_fixed = '''
SELECT member_id, nickname 
FROM demo_members
WHERE member_id NOT IN (
    SELECT member_id 
    FROM demo_bar_orders 
    WHERE member_id IS NOT NULL
);
'''
res_fixed = pd.read_sql_query(query_fixed, conn)
display(res_fixed)
print(f"Строк возвращено: {len(res_fixed)} (Все найдены корректно!)")


1. Опасный запрос (без фильтрации NULL) возвращает:
Empty DataFrame
Columns: [member_id, nickname]
Index: []
Строк возвращено: 0 (Хотя клиенты Ghost, Frost и Phoenix в баре ничего не брали!)

2. Исправленный запрос с фильтром WHERE member_id IS NOT NULL:
   member_id nickname
0          4    Ghost
1          5    Frost
2          6  Phoenix
Строк возвращено: 3 (Все найдены корректно!)


---
## Тема 5. Коррелированные подзапросы (Correlated Subqueries)

### 📌 На доску
```sql
-- Найти сессии, которые длились дольше, чем средняя сессия НА ЭТОМ ЖЕ компьютере
SELECT s1.session_id, s1.pc_id, s1.hours_played
FROM demo_sessions s1
WHERE s1.hours_played > (
    SELECT AVG(s2.hours_played)
    FROM demo_sessions s2
    WHERE s2.pc_id = s1.pc_id -- Связь с внешним запросом!
);
```

### 💡 Зачем это нужно
* Когда объект нужно сравнить **не со всей таблицей целиком**, а со своей собственной группой/категорией (например, сравнить зарплату сотрудника со средней зарплатой в его отделе).

### ⚙️ Как это работает
* В отличие от обычного подзапроса, коррелированный подзапрос **не может выполниться отдельно от внешнего**.
* Для каждой строки таблицы `s1` СУБД:
  1. Берет текущий `s1.pc_id` (например, 101);
  2. Подставляет его в подзапрос: `WHERE s2.pc_id = 101`;
  3. Вычисляет локальное среднее по ПК 101;
  4. Сравнивает текущие часы `s1.hours_played` с этим локальным средним.


In [5]:
# Демонстрация 5: Коррелированный подзапрос в действии
query_corr = '''
SELECT 
    s1.session_id,
    s1.pc_id,
    s1.game_title,
    s1.hours_played,
    ROUND((SELECT AVG(s2.hours_played) FROM demo_sessions s2 WHERE s2.pc_id = s1.pc_id), 2) AS avg_hours_for_this_pc
FROM demo_sessions s1
WHERE s1.hours_played > (
    SELECT AVG(s2.hours_played)
    FROM demo_sessions s2
    WHERE s2.pc_id = s1.pc_id
);
'''
pd.read_sql_query(query_corr, conn)


session_id,pc_id,game_title,hours_played,avg_hours_for_this_pc
2,103,Cyberpunk 2077,3.0,2.75
4,102,Counter-Strike 2,4.0,2.75
6,101,Counter-Strike 2,5.0,2.67


---
## Тема 6. Предикаты существования: `EXISTS` и `NOT EXISTS`

### 📌 На доску
```sql
-- Клиенты, заказывавшие горячее питание
SELECT m.member_id, m.nickname
FROM demo_members m
WHERE EXISTS (
    SELECT 1 
    FROM demo_bar_orders b 
    WHERE b.member_id = m.member_id 
      AND b.category = 'Горячее'
);
```

### 💡 Зачем это нужно
* `EXISTS` отвечает строго на вопрос: **«Существует ли хотя бы одна запись?»** (Возвращает `TRUE` или `FALSE`).
* Ему не нужно сканировать миллион строк до конца: как только встретилась **первая подходящая строка**, СУБД немедленно останавливает поиск для данного клиента (**short-circuit evaluation**).
* Поэтому в блоке `SELECT` подзапроса принято писать `SELECT 1` — СУБД вообще не читает данные колонок из памяти, проверяя только факт наличия записи в индексе.
* `NOT EXISTS` абсолютно безопасен при наличии `NULL`. Он никогда не страдает от трехзначной логики.


In [6]:
# Демонстрация 6: Проверка существования через EXISTS и поиск сирот через NOT EXISTS

print("1. Клиенты, бравшие горячее питание (EXISTS):")
query_exists = '''
SELECT m.member_id, m.nickname, m.city
FROM demo_members m
WHERE EXISTS (
    SELECT 1 
    FROM demo_bar_orders b 
    WHERE b.member_id = m.member_id AND b.category = 'Горячее'
);
'''
display(pd.read_sql_query(query_exists, conn))

print("\n2. Компьютеры, на которых не было ни одной игры (NOT EXISTS):")
query_not_exists = '''
SELECT p.pc_id, p.zone_name, p.hourly_rate
FROM demo_pcs p
WHERE NOT EXISTS (
    SELECT 1 
    FROM demo_sessions s 
    WHERE s.pc_id = p.pc_id
);
'''
display(pd.read_sql_query(query_not_exists, conn))


1. Клиенты, бравшие горячее питание (EXISTS):
   member_id nickname    city
0          1   Shadow  Москва
1          3      Neo  Москва

2. Компьютеры, на которых не было ни одной игры (NOT EXISTS):
   pc_id zone_name  hourly_rate
0    104  Bootcamp          250


---
## Тема 7. Подзапросы в `SELECT` («Клиент 360») и производные таблицы в `FROM`

### 📌 На доску
```sql
-- 1. Подзапрос в SELECT (Вычисленная колонка для витрины)
SELECT 
    m.nickname,
    (SELECT COUNT(*) FROM demo_sessions s WHERE s.member_id = m.member_id) AS total_sessions,
    (SELECT IFNULL(SUM(total_amount), 0) FROM demo_bar_orders b WHERE b.member_id = m.member_id) AS bar_spent
FROM demo_members m;

-- 2. Подзапрос в FROM (Агрегация над агрегацией)
SELECT ROUND(AVG(sessions_per_user), 2) AS avg_sessions_per_active_client
FROM (
    SELECT member_id, COUNT(*) AS sessions_per_user
    FROM demo_sessions
    WHERE member_id IS NOT NULL
    GROUP BY member_id
) AS client_activity; -- Алиас AS обязателен!
```

### 💡 Зачем это нужно
* **В `SELECT`:** сборка клиентских витрин (Customer 360), когда данные нужно подтянуть из 4-5 таблиц без риска размножения строк декартовым произведением от множественных `LEFT JOIN`.
* **В `FROM`:** расчет «агрегатов от агрегатов». Например, мы не можем написать `AVG(COUNT(*))` в одном запросе — SQL запрещает вложенные агрегаты. Решение — посчитать `COUNT(*)` во внутреннем запросе в `FROM`, а внешний запрос посчитает от него `AVG()`.


In [7]:
# Демонстрация 7: Аналитическая витрина в SELECT и агрегат над агрегатом в FROM

print("1. Витрина гостя (подзапросы в SELECT):")
query_select_sub = '''
SELECT 
    m.member_id,
    m.nickname,
    m.loyalty_tier,
    (SELECT COUNT(*) FROM demo_sessions s WHERE s.member_id = m.member_id) AS sessions_count,
    (SELECT IFNULL(SUM(total_paid), 0) FROM demo_sessions s WHERE s.member_id = m.member_id) AS games_rub,
    (SELECT IFNULL(SUM(total_amount), 0) FROM demo_bar_orders b WHERE b.member_id = m.member_id) AS bar_rub
FROM demo_members m
ORDER BY games_rub DESC;
'''
display(pd.read_sql_query(query_select_sub, conn))

print("\n2. Среднее число сессий на активного игрока (производная таблица в FROM):")
query_from_sub = '''
SELECT 
    ROUND(AVG(player_sessions), 2) AS avg_sessions,
    MIN(player_sessions) AS min_sessions,
    MAX(player_sessions) AS max_sessions
FROM (
    SELECT member_id, COUNT(*) AS player_sessions
    FROM demo_sessions
    WHERE member_id IS NOT NULL
    GROUP BY member_id
) AS player_stats;
'''
display(pd.read_sql_query(query_from_sub, conn))


1. Витрина гостя (подзапросы в SELECT):
   member_id nickname loyalty_tier  sessions_count  games_rub  bar_rub
0          1   Shadow         Gold               2       1500      600
1          3      Neo     Platinum               1       1000      800
2          2    Viper       Silver               2        750      180
3          6  Phoenix       Silver               1        750        0
4          4    Ghost       Bronze               0          0        0
5          5    Frost       Bronze               0          0        0

2. Среднее число сессий на активного игрока (производная таблица в FROM):
   avg_sessions  min_sessions  max_sessions
0           1.5             1             2


---
## Тема 8. Операции над множествами (`UNION`, `UNION ALL`, `INTERSECT`, `EXCEPT`)

### 📌 На доску
| Оператор | Математический смысл | Дедупликация | Скорость |
| :--- | :--- | :--- | :--- |
| **`UNION`** | Объединение $A \cup B$ | Да (удаляет повторы) | Медленнее (требует сортировку) |
| **`UNION ALL`** | Конкатенация $A + B$ | Нет (сохраняет все) | Максимальная (без сортировки) |
| **`INTERSECT`** | Пересечение $A \cap B$ | Да (только общие элементы) | Требует сортировку |
| **`EXCEPT`** | Разность $A \setminus B$ | Да (есть в $A$, но нет в $B$) | Требует сортировку |

### ⚙️ Железные правила операций над множествами:
1. Количество столбцов во всех объединяемых запросах должно быть **абсолютно одинаковым**.
2. Позиционные типы данных должны быть совместимы.
3. Названия столбцов берутся из **первого** запроса.


In [8]:
# Демонстрация 8: Операции над множествами

print("1. INTERSECT: Клиенты, которые И играли на ПК, И заказывали в баре:")
query_intersect = '''
SELECT member_id FROM demo_sessions WHERE member_id IS NOT NULL
INTERSECT
SELECT member_id FROM demo_bar_orders WHERE member_id IS NOT NULL;
'''
intersect_ids = pd.read_sql_query(query_intersect, conn)
display(intersect_ids)

print("\n2. EXCEPT: Игроки, которые НИ РАЗУ ничего не купили в баре:")
query_except = '''
SELECT member_id FROM demo_sessions WHERE member_id IS NOT NULL
EXCEPT
SELECT member_id FROM demo_bar_orders WHERE member_id IS NOT NULL;
'''
except_ids = pd.read_sql_query(query_except, conn)
display(except_ids)

print("\n3. UNION vs UNION ALL:")
q_union = "SELECT member_id FROM demo_sessions WHERE member_id IS NOT NULL UNION SELECT member_id FROM demo_bar_orders WHERE member_id IS NOT NULL"
q_union_all = "SELECT member_id FROM demo_sessions WHERE member_id IS NOT NULL UNION ALL SELECT member_id FROM demo_bar_orders WHERE member_id IS NOT NULL"
print(f"Строк с UNION (без дубликатов): {len(pd.read_sql_query(q_union, conn))}")
print(f"Строк с UNION ALL (с дубликатами): {len(pd.read_sql_query(q_union_all, conn))}")


1. INTERSECT: Клиенты, которые И играли на ПК, И заказывали в баре:
   member_id
0          1
1          2
2          3

2. EXCEPT: Игроки, которые НИ РАЗУ ничего не купили в баре:
   member_id
0          6

3. UNION vs UNION ALL:
Строк с UNION (без дубликатов): 4
Строк с UNION ALL (с дубликатами): 11


---
## Итоговое резюме: Как выбирать инструмент?

| Задача | Рекомендуемый инструмент | Почему? |
| :--- | :--- | :--- |
| Нужно вывести столбцы из двух и более таблиц | **`JOIN`** | Подзапрос в `SELECT` вернет только 1 столбец, а `JOIN` соединяет целые строки. |
| Фильтрация по наличию факта без вывода полей | **`EXISTS`** | Работает со short-circuit, не сканирует лишние строки и не размножает результат. |
| Поиск объектов, которых НЕТ в связанной таблице | **`NOT EXISTS`** | Безопасен к `NULL`, быстрее, чем `LEFT JOIN ... WHERE IS NULL`. |
| Сравнение с порогом (среднее, максимум, минимум) | **Скалярный подзапрос в `WHERE`** | Вычисляет динамическую планку прямо во время выполнения запроса. |
| Расчет среднего от количества или суммы | **Производная таблица в `FROM`** | Единственный чистый способ обойти запрет вложенных агрегатных функций. |
| Сквозная финансовая лента из разных источников | **`UNION ALL`** | Объединяет разнородные потоки платежей в единую хронологию. |
